# Retrieval Models Aren't Tool-Savvy (TOOLRET) - Hands-on Build

**Date:** 2026-06-09  
**Explainer:** [../toolret.md](../toolret.md)  
**Paper:** [arXiv:2503.01763](https://arxiv.org/abs/2503.01763)

---

## Problem

A wealth-management copilot must pick the right analytics tools from a **heterogeneous catalog** -
Web-API quote feeds, Python risk functions, and in-house app actions - for each advisor query.
But advisor questions share almost no vocabulary with the tools they need ("trim my tech
exposure" vs `rebalance_portfolio` + `sector_allocation`), and most tasks need **several** tools
at once. Build a miniature **TOOLRET**: assemble a mixed-format tool corpus + multi-target
retrieval tasks, retrieve with and without instructions, score **Completeness@k / nDCG@k**, prove
that bad retrieval **tanks the agent's task pass rate**, then close the gap with instruction
augmentation + hard-negative mining.

> [!WARNING]
> **Synthetic / public data only.** Decision-support, not advice. No real client data or MNPI.

In [7]:
import os
import re
import math
import json
import time
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Optional, Literal, Set

import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field

# LangChain messages / prompts / tools
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel

# LangChain model providers + embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_mistralai import ChatMistralAI

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Send

# Verify required API keys
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'
assert os.getenv('MISTRAL_API_KEY'), 'MISTRAL_API_KEY not set'
assert os.getenv('LANGSMITH_API_KEY'), 'LANGSMITH_API_KEY not set'

In [8]:
# LLM instances - one per provider
llmOpenAI = ChatOpenAI(model='gpt-5.4-mini', temperature=0.0)
llmAnthropic = ChatAnthropic(model='claude-haiku-4-5', temperature=0.0)
llmMistral = ChatMistralAI(model='mistral-medium-3-5', temperature=0.0)

# Embedder for the (out-of-the-box) retriever - TOOLRET evaluates retrievers WITHOUT fine-tuning first.
embedder = OpenAIEmbeddings(model='text-embedding-3-small')

# gpt-4o stand-in used as the 'powerful LLM' instruction generator in the paper.
llmInstructionGen = llmOpenAI

## Synthetic data - a heterogeneous WM tool corpus + multi-target retrieval tasks

A tiny stand-in for TOOLRET's 43k corpus. Each tool carries a `format` tag (`web_api` /
`code_function` / `customized_app`) so we can mirror the paper's three subsets. The `GOLDEN`
tasks are deliberately **multi-target** (avg ~2 tools, like the paper's 2.17) and worded with
**low lexical overlap** to the tool docs - so lexical matching alone should struggle.

In [21]:
# Heterogeneous corpus: three documentation FORMATS, like TOOLRET's Web API / Code / Customized App.
TOOL_CORPUS: List[Dict[str, Any]] = [
    # --- web_api (structured JSON / OpenAPI-style) ---
    {'id': 'T01', 'name': 'rebalance_portfolio', 'format': 'web_api', 'doc': '{"endpoint": "/portfolio/rebalance", "desc": "Rebalance an account back to target asset-allocation weights", "params": {"account_id": "str", "target_weights": "dict"}}'},
    {'id': 'T02', 'name': 'sector_allocation', 'format': 'web_api', 'doc': '{"endpoint": "/portfolio/sectors", "desc": "Break down portfolio market value by GICS sector", "params": {"account_id": "str"}}'},
    {'id': 'T03', 'name': 'fx_convert', 'format': 'web_api', 'doc': '{"endpoint": "/fx/convert", "desc": "Convert an amount between two currencies at the latest rate", "params": {"amount": "float", "from_ccy": "str", "to_ccy": "str"}}'},
    {'id': 'T04', 'name': 'get_quote', 'format': 'web_api', 'doc': '{"endpoint": "/market/quote", "desc": "Fetch the latest price quote for a ticker symbol", "params": {"ticker": "str"}}'},
    {'id': 'T05', 'name': 'var_estimate', 'format': 'web_api', 'doc': '{"endpoint": "/risk/var", "desc": "Estimate 1-day 95% Value-at-Risk for the portfolio", "params": {"account_id": "str"}}'},
    # --- code_function (function-level snippets) ---
    {'id': 'T06', 'name': 'compute_npv', 'format': 'code_function', 'doc': 'def compute_npv(cashflows: list[float], rate: float) -> float:\n    """Net present value of a cash-flow stream at a discount rate."""'},
    {'id': 'T07', 'name': 'sharpe_ratio', 'format': 'code_function', 'doc': 'def sharpe_ratio(returns: list[float], rf_rate: float) -> float:\n    """Annualized Sharpe ratio of a return series."""'},
    {'id': 'T08', 'name': 'factor_exposure', 'format': 'code_function', 'doc': 'def factor_exposure(account_id: str) -> dict:\n    """Exposure of holdings to value, momentum, size, quality factors."""'},
    {'id': 'T09', 'name': 'options_greeks', 'format': 'code_function', 'doc': 'def options_greeks(ticker: str, strike: float, expiry: str) -> dict:\n    """Delta, gamma, theta, vega for an options position."""'},
    # --- customized_app (free-form natural language) ---
    {'id': 'T10', 'name': 'tax_loss_harvest', 'format': 'customized_app', 'doc': 'Scans an account for tax lots sitting at an unrealized loss and proposes which to sell to offset capital gains.'},
    {'id': 'T11', 'name': 'risk_attribution', 'format': 'customized_app', 'doc': 'Explains where a portfolio\u2019s risk is coming from, decomposing it across asset classes and factors.'},
    {'id': 'T12', 'name': 'dividend_forecast', 'format': 'customized_app', 'doc': 'Projects the expected dividend income an account will receive over the next twelve months.'},
    {'id': 'T13', 'name': 'esg_screen', 'format': 'customized_app', 'doc': 'Flags holdings that fail a client\u2019s sustainability preferences and suggests compliant replacements.'},
    {'id': 'T14', 'name': 'cash_sweep', 'format': 'customized_app', 'doc': 'Moves idle cash above a threshold into a money-market sweep vehicle.'},
]

# Multi-target retrieval tasks: low lexical overlap with tool docs, several targets each.
GOLDEN: List[Dict[str, Any]] = [
    {'query': 'Trim my overweight tech and put the proceeds back on target.', 'targets': {'T02', 'T01'}},
    {'query': 'I want to lower my downside and understand where the danger is concentrated.', 'targets': {'T05', 'T11'}},
    {'query': 'Lock in some losses for taxes and tell me what income I will still collect this year.', 'targets': {'T10', 'T12'}},
    {'query': 'Whats this deal worth today, and show it to me in euros?', 'targets': {'T06', 'T03'}},
]

# 3 handcrafted SEED instructions (the paper used 100) to prime the target-aware generator.
SEED_INSTRUCTIONS: List[str] = [
    'Retrieve tools that adjust a portfolio\u2019s holdings to match a desired allocation.',
    'Retrieve tools that quantify or explain investment risk for an account.',
    'Retrieve tools that handle tax-aware actions and projected cash income for an account.',
]

print(f'corpus size = {len(TOOL_CORPUS)} tools across', sorted({t["format"] for t in TOOL_CORPUS}))
print(f'retrieval tasks = {len(GOLDEN)} (avg targets = {np.mean([len(g["targets"]) for g in GOLDEN]):.2f})')

corpus size = 14 tools across ['code_function', 'customized_app', 'web_api']
retrieval tasks = 4 (avg targets = 2.00)


## Coverage Map - complete every section below to cover the whole paper

| # | Paper aspect | Role in this build | Section |
|---|---|---|---|
| 1 | **Heterogeneous tool corpus** (Web API / Code / Customized App) | A mixed-format catalog with a unified embeddable text view | S1 |
| 2 | **Retrieval tasks** (query → multi-target tools, MTEB-style) | The `GOLDEN` tasks: several correct tools per query | S2 |
| 3 | **Lexical overlap analysis** (ROUGE-L query↔target) | Show overlap is ~0 → semantic retrieval is mandatory | S3 |
| 4 | **Target-aware instruction construction** | LLM writes a relevance-criteria instruction per query from seeds | S4 |
| 5 | **Dense retriever** (embed → cosine → top-k) | The core 🎯 Select step under test | S5 |
| 6 | **Sparse BM25 baseline** | Reproduce "strong dense models can lose to lexical BM25" | S6 |
| 7 | **Two settings: w/o inst. vs w/ inst.** | Retrieve on query alone vs query⊕instruction | S7 |
| 8 | **Metrics: nDCG@k, Recall@k, Precision@k, Completeness@k** | Score retrieval; Completeness@k is all-or-nothing | S8 |
| 9 | **Re-ranking** (cross-encoder / LLM rerank) | Re-score the shortlist; observe it may NOT help | S9 |
| 10 | **Downstream pass-rate vs oracle** | Feed retrieved vs oracle tools to an agent; measure pass rate | S10 |
| 11 | **TOOLRET-train** (hard-negative mining + instruction tuning) | Build contrastive training examples; show the w/ inst. lift | S11 |
| 12 | **Wire-up into one graph** | Assemble retrieve → rerank → agent → pass-rate as a LangGraph | S12 |

### S1 . Heterogeneous tool corpus

**What:** TOOLRET merges 30+ datasets into one corpus spanning three documentation formats and
reformats everything into a single IR-style schema. **Maps to:** giving our mixed `web_api` /
`code_function` / `customized_app` tools one unified text view that any retriever can embed.

In [22]:
def tool_text(tool: Dict[str, Any]) -> str:
    return f"{tool['name']} :: {tool['doc']}"

def corpus_by_format(corpus: List[Dict] = TOOL_CORPUS) -> Dict[str, List[Dict]]:
    # TODO: group the corpus into the three subsets so you can report per-format retrieval later.
    grouped : Dict[str,List[Dict]] = {}
    for t in corpus:
        grouped.setdefault(t['format'],[]).append(t)
    return grouped

ID2TOOL: Dict[str, Dict] = {t['id']: t for t in TOOL_CORPUS}

### S2 . Retrieval tasks (query → multi-target tools)

**What:** each TOOLRET task is a query paired with its set of target tool IDs (labels), like
MTEB. Crucially, a task usually has **several** targets (avg 2.17). **Maps to:** the `GOLDEN`
list; here we add a tiny validator so each task is well-formed before scoring.

In [23]:
def validate_tasks(golden: List[Dict] = GOLDEN, id2tool: Dict = ID2TOOL) -> None:
    # Assert each task is well-formed, then confirm the multi-target property.
    for i, task in enumerate(golden):
        targets = task['targets']
        # (a) every task must have at least one target
        assert len(targets) >= 1, f'task {i} ({task["query"]!r}) has no targets'
        # (b) every target ID must actually exist in the corpus
        unknown = set(targets) - id2tool.keys()
        assert not unknown, f'task {i} references unknown tool IDs: {unknown}'

    avg = np.mean([len(t['targets']) for t in golden])
    # (c) the benchmark is meant to be multi-target, like the paper's 2.17
    assert avg > 1, f'expected multi-target tasks but avg = {avg:.2f}'
    print(f'OK: {len(golden)} tasks well-formed, all targets in corpus, avg targets/query = {avg:.2f}')

validate_tasks()


OK: 4 tasks well-formed, all targets in corpus, avg targets/query = 2.00


### S3 . Lexical overlap analysis (ROUGE-L)

**What:** the paper's headline difficulty stat - query↔target ROUGE-L of just **0.06** (vs 0.31
for NQ) - is *why* lexical retrieval fails and semantics are required. **Maps to:** computing the
same overlap on our tasks to confirm we built a genuinely hard, low-overlap benchmark.

In [25]:
from difflib import SequenceMatcher

def rouge_l(a: str, b: str) -> float:
    # TODO: longest-common-subsequence F-measure between token lists of a and b (0..1).
    tokens_a = a.split()
    tokens_b = b.split()
    if not tokens_a or not tokens_b:
        return 0.0
    return SequenceMatcher(None,tokens_a,tokens_b,autojunk=False).ratio()

def avg_query_target_overlap(golden: List[Dict] = GOLDEN) -> float:
    # TODO: for each task, mean ROUGE-L(query, tool_text(target)) over its targets;
    #       return the mean across tasks. Expect a LOW number (close to the paper's 0.06).
    list_rouge_l = []
    for item in golden : 
        query = item['query']
        targets = item ['targets']
        rouge_l_score = np.mean([rouge_l(query,tool_text(ID2TOOL[target])) for target in targets])
        list_rouge_l.append(rouge_l_score)

    return np.mean(list_rouge_l)

    
print('avg query<->target ROUGE-L =', avg_query_target_overlap())

avg query<->target ROUGE-L = 0.05385462238910515


### S4 . Target-aware instruction construction

**What:** TOOLRET supplements each query with an LLM-generated **instruction** that states the
relevance criteria (bridging the query intent and the target tools), seeded from handcrafted
examples. **Maps to:** generating one instruction per `GOLDEN` query via in-context learning
over `SEED_INSTRUCTIONS`.

In [34]:
from concurrent.futures import ThreadPoolExecutor, as_completed
def generate_instruction(query: str, target_docs: List[str], seeds: List[str] = SEED_INSTRUCTIONS, llm=llmInstructionGen) -> str:
    # TODO: prompt the LLM with the seeds as in-context examples + the query (and target docs, since
    #       the strategy is 'target-aware') to emit ONE 'Retrieve tools that ...' relevance instruction.

    target_docs_content = [tool_text(ID2TOOL[target]) for target in target_docs]
    tool_name_list = [ID2TOOL[target]['name'] for target in target_docs]
    n = len(target_docs)
    seed_examples = "\n".join(f"  - {s}" for s in seeds)
    target_docs_formatted = "\n".join(f"  - {doc}" for doc in target_docs_content)
    
    messages= [ SystemMessage(content=(
            "You are an instruction-generation specialist. "
            "You write concise relevance instructions that bridge a user's query intent "
            "with target tools. Instructions must start with 'Retrieve tools that ...'. "
            "Output ONLY the instruction — one sentence, no preamble, no explanation."
        )),
        HumanMessage(content=f"""Query: {query}
                Target Tools ({n} total):
                {"\n".join(f"   - {name}" for name in tool_name_list)}
            CRITICAL CONSTRAINT: These {n} documents represent {n} distinct capabilities. 
            Your single instruction MUST reference the function of EACH one — do not merge 
            or omit any. 
            Every tool name listed above ({target_docs_formatted}) 
            must have its capability reflected in the instruction.
            Reference Examples (style only — do NOT copy):
            {seed_examples}
            Now generate ONE instruction that covers ALL {n} tools.""")
        ]

    result = llm.invoke(messages)

    return result.content

def build_instructions(golden: List[Dict] = GOLDEN) -> Dict[str, str]:
    # TODO: return {query: instruction} for every task. Optionally add a quality check
    #       (does the instruction mention the target tools' function?) mirroring the paper's review.
    instructions_map = {}
    with ThreadPoolExecutor(max_workers=len(golden)) as executor:
        futures_to_instructions = {executor.submit(generate_instruction,task['query'],list(task['targets'])): task['query'] for task in golden }
        for future in as_completed(futures_to_instructions):
            query = futures_to_instructions[future]
            try:
                instruction = future.result()
                instructions_map[query] = instruction
            except Exception as e:
                print(f"Failed for query: {query!r} - {e}")
    
    return instructions_map

    

INSTRUCTIONS = build_instructions()

In [36]:
INSTRUCTIONS

{'I want to lower my downside and understand where the danger is concentrated.': 'Retrieve tools that estimate 1-day 95% Value-at-Risk to lower downside and attribute portfolio risk across asset classes and factors to identify where danger is concentrated.',
 'Trim my overweight tech and put the proceeds back on target.': 'Retrieve tools that show a portfolio’s sector breakdown and rebalance an account back to target asset-allocation weights.',
 'Whats this deal worth today, and show it to me in euros?': 'Retrieve tools that compute the deal’s present value from its cash flows at a discount rate and then convert that value into euros at the latest exchange rate.',
 'Lock in some losses for taxes and tell me what income I will still collect this year.': 'Retrieve tools that scan an account for tax lots at unrealized losses to propose tax-loss harvesting sales and project the expected dividend income the account will still collect over the next twelve months.'}

llmo

### S5 . Dense retriever (the 🎯 Select step under test)

**What:** the off-the-shelf neural retriever - embed the query and every tool, rank by cosine
similarity, take top-k. This is exactly what TOOLRET shows is surprisingly weak on tools.
**Maps to:** our `DenseRetriever` over the WM corpus.

In [ ]:
class DenseRetriever:
    """Embed the corpus once; rank tools by cosine similarity to a query string."""

    def __init__(self, corpus: List[Dict] = TOOL_CORPUS, embedder=embedder):
        self.corpus = corpus
        self.embedder = embedder
        self.ids: List[str] = [t['id'] for t in corpus]
        self.matrix: Optional[np.ndarray] = None  # (N, d) tool embeddings

    def index(self) -> None:
        # TODO: embed tool_text(t) for every tool; store an L2-normalized matrix in self.matrix.
        raise NotImplementedError

    def search(self, query: str, k: int = 5) -> List[str]:
        # TODO: embed query, cosine vs self.matrix, return top-k tool IDs (highest first).
        raise NotImplementedError

### S6 . Sparse BM25 baseline

**What:** TOOLRET's most humbling result - a strong dense model (ColBERT) can *lose* to plain
lexical BM25. You need the lexical baseline to reproduce that comparison. **Maps to:** a tiny
BM25 scorer over the same corpus, ranked the same way.

In [ ]:
class BM25Retriever:
    """Classic lexical BM25 over tokenized tool_text."""

    def __init__(self, corpus: List[Dict] = TOOL_CORPUS, k1: float = 1.5, b: float = 0.75):
        self.corpus = corpus
        self.k1, self.b = k1, b
        # TODO: tokenize each tool_text, build term frequencies, doc lengths, and IDF.

    def search(self, query: str, k: int = 5) -> List[str]:
        # TODO: score each tool with the BM25 formula vs the query tokens; return top-k tool IDs.
        raise NotImplementedError

### S7 . Two settings - w/o inst. vs w/ inst.

**What:** TOOLRET evaluates every retriever twice: query alone, and query⊕instruction. The
instruction reliably lifts scores (most for instruction-tuned models). **Maps to:** a switch that
feeds either `query` or `query + ' ' + instruction` into a retriever.

In [ ]:
def retrieve(retriever, task: Dict, k: int, instructions: Optional[Dict[str, str]] = None, use_instruction: bool = False) -> List[str]:
    # TODO: build the input string - query alone, or query + ' ' + instructions[query] when use_instruction;
    #       return retriever.search(input_str, k).
    raise NotImplementedError

### S8 . Metrics - nDCG@k, Recall@k, Precision@k, Completeness@k

**What:** the evaluation core. **Completeness@k** is TOOLRET's signature metric: it is 1 only if
**all** target tools are inside the top-k, else 0 - brutal for multi-target tasks. **Maps to:**
scoring a retriever's ranked IDs against each task's `targets`.

In [ ]:
def recall_at_k(retrieved: List[str], targets: Set[str], k: int) -> float:
    # TODO: |top-k & targets| / |targets|
    raise NotImplementedError

def precision_at_k(retrieved: List[str], targets: Set[str], k: int) -> float:
    # TODO: |top-k & targets| / k
    raise NotImplementedError

def completeness_at_k(retrieved: List[str], targets: Set[str], k: int) -> float:
    # TODO: 1.0 if targets is a subset of top-k else 0.0  (all-or-nothing)
    raise NotImplementedError

def ndcg_at_k(retrieved: List[str], targets: Set[str], k: int) -> float:
    # TODO: binary-relevance DCG over top-k divided by ideal DCG; return 0..1.
    raise NotImplementedError

def evaluate(retriever, golden: List[Dict] = GOLDEN, k: int = 5, instructions=None, use_instruction: bool = False) -> Dict[str, float]:
    # TODO: average each metric across tasks; return {'nDCG': .., 'Recall': .., 'Precision': .., 'Completeness': ..}.
    raise NotImplementedError

# Reproduce the paper's shape:
# dense = DenseRetriever(); dense.index()
# bm25 = BM25Retriever()
# print('dense', evaluate(dense)); print('bm25', evaluate(bm25))   # is dense actually winning?

### S9 . Re-ranking (and watching it not help)

**What:** the classic upgrade - take the retriever's shortlist and re-score it with a heavier
cross-encoder / LLM judge. TOOLRET found this gives *limited or negative* gains on tools.
**Maps to:** an LLM that re-orders the dense shortlist; compare metrics before vs after.

In [ ]:
def llm_rerank(query: str, candidate_ids: List[str], top_k: int, id2tool: Dict = ID2TOOL, llm=llmOpenAI) -> List[str]:
    # TODO: show the LLM the query + each candidate's tool_text; ask it to return the top_k IDs
    #       most relevant, in order. Then compare evaluate(before) vs evaluate(after).
    raise NotImplementedError

### S10 . Downstream pass-rate vs oracle

**What:** the payoff experiment - feed the agent the **retrieved** toolset instead of the
**oracle** (golden) toolset and watch the end-to-end pass rate drop ~10-11 points. **Maps to:** a
toy tool-use agent that 'passes' a task only if it can call all the right tools - which it can't
if retrieval missed one.

In [ ]:
def task_pass(available_ids: List[str], targets: Set[str]) -> float:
    # TODO: toy pass criterion - 1.0 if the agent CAN call every target tool (targets subset of available),
    #       else 0.0. (Optionally bind the tools to an LLM and check it selects them.)
    raise NotImplementedError

def pass_rate(retriever, golden: List[Dict] = GOLDEN, k: int = 5, instructions=None, use_instruction: bool = False) -> float:
    # TODO: ORACLE pass rate = mean(task_pass(list(targets), targets)) == 1.0 by construction.
    #       RETRIEVED pass rate = mean(task_pass(retrieve(...k...), targets)). Print the gap (the paper's -11).
    raise NotImplementedError

### S11 . TOOLRET-train - hard-negative mining + instruction tuning

**What:** the cure. Each training example = query + instruction + target tools + **10 hard
negatives** mined by the retriever (top-ranked wrong tools), used in a contrastive loss. Full
fine-tuning is out of scope here, so we (a) **construct** the training examples and (b) prove the
*mechanism* of the gain by showing w/ inst. beats w/o inst. on our metrics.

In [ ]:
def mine_hard_negatives(retriever, task: Dict, n: int = 10) -> List[str]:
    # TODO: retrieve top-(n + |targets|) IDs, drop the true targets, keep the first n wrong ones.
    raise NotImplementedError

def build_training_example(retriever, task: Dict, instructions: Dict[str, str], n_neg: int = 10) -> Dict:
    # TODO: return {'query', 'instruction', 'positives': list(targets), 'negatives': mine_hard_negatives(...)}
    #       - the exact (q, I, T+, T-) contrastive record TOOLRET-train trains on.
    raise NotImplementedError

# Demonstrate the lever WITHOUT fine-tuning: instruction-augmented retrieval should beat bare query.
# dense = DenseRetriever(); dense.index(); INSTRUCTIONS = build_instructions()
# print('w/o inst.', evaluate(dense, k=5, instructions=INSTRUCTIONS, use_instruction=False))
# print('w/  inst.', evaluate(dense, k=5, instructions=INSTRUCTIONS, use_instruction=True))
# train_set = [build_training_example(dense, t, INSTRUCTIONS) for t in GOLDEN]

## S12 . Wire-up - assemble retrieve → rerank → agent → pass-rate as one graph

Connect the pieces into a single runnable LangGraph: a query enters, gets an instruction,
retrieves a toolset (w/ inst.), optionally reranks, hands the toolset to the agent, and records
whether the task passed - so you can run the whole TOOLRET loop end-to-end on any query.

In [ ]:
class ToolRetState(TypedDict):
    query: str
    targets: Set[str]
    instruction: str
    candidates: List[str]
    final_tools: List[str]
    passed: float
    messages: Annotated[List[BaseMessage], add_messages]

# TODO: define node functions wrapping S4-S10, e.g.
# def n_instruct(state): return {'instruction': generate_instruction(state['query'], ...)}
# def n_retrieve(state): return {'candidates': retrieve(dense, state, k=8, instructions={...}, use_instruction=True)}
# def n_rerank(state):   return {'final_tools': llm_rerank(state['query'], state['candidates'], top_k=5)}
# def n_passcheck(state):return {'passed': task_pass(state['final_tools'], state['targets'])}

# TODO: build the graph
# g = StateGraph(ToolRetState)
# g.add_node('instruct', n_instruct); g.add_node('retrieve', n_retrieve)
# g.add_node('rerank', n_rerank); g.add_node('passcheck', n_passcheck)
# g.add_edge(START, 'instruct'); g.add_edge('instruct', 'retrieve')
# g.add_edge('retrieve', 'rerank'); g.add_edge('rerank', 'passcheck'); g.add_edge('passcheck', END)
# app = g.compile()
# app.invoke({'query': GOLDEN[0]['query'], 'targets': GOLDEN[0]['targets'], 'messages': []})

## Validation - run end-to-end, then tick every box

Run the pipeline on each `GOLDEN` task and confirm each aspect works. Completing this checklist
== you've reconstructed the whole paper.

- [ ] **S1** Corpus unified into one embeddable text view across all 3 formats
- [ ] **S2** Tasks validated; avg targets/query > 1 (multi-target like the paper)
- [ ] **S3** Query↔target ROUGE-L computed and is LOW (semantic retrieval required)
- [ ] **S4** A target-aware instruction is generated per query from the seeds
- [ ] **S5** Dense retriever indexes the corpus and returns top-k IDs
- [ ] **S6** BM25 baseline returns top-k IDs over the same corpus
- [ ] **S7** Retrieval runs in both w/o inst. and w/ inst. settings
- [ ] **S8** nDCG@k, Recall@k, Precision@k, **Completeness@k** all computed
- [ ] **S9** Re-ranking applied; you compared metrics before vs after (did it help?)
- [ ] **S10** Retrieved-vs-oracle **pass-rate gap** measured (reproduce the ~-11 effect)
- [ ] **S11** Hard negatives mined; (q, I, T+, T-) training examples built; w/ inst. beats w/o inst.
- [ ] **S12** All phases run as one LangGraph on every GOLDEN task